# Experiment 5: Subword Level Tokenization and POS Tagging

This notebook implements all parts of Experiment 5:
- 5.1 BPE-based tokenization: pretrained and trained from the dataset
- 5.2 SentencePiece tokenization: pretrained and trained from the dataset
- 5.3 POS tagging: SpaCy and NLTK
- 5.4 POS tagging with token frequency using SpaCy

**Dataset:** Upload the provided `.txt` corpus when prompted.

## Dataset Import
Upload the dataset from your computer in Google Colab.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

with open(file_name, 'r', encoding='utf-8') as f:
    text = f.read()

print('Dataset loaded successfully!')
print('File:', file_name)
print('Number of characters:', len(text))

print('\nFirst 1000 characters:\n')
print(text[:1000])

Saving input_sub_word_data.txt to input_sub_word_data.txt
Dataset loaded successfully!
File: input_sub_word_data.txt
Number of characters: 6682

First 1000 characters:

Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is simple and
easy to understand, but it may have difficulty handling rare and unknown
words.

Subword-level tokenization provides an effective solution to the problem
of rare words. Instead of representing every complete word as a separate
token, a subword tokenizer represents a word using smaller meaningful units.
For example, the

## 5.1(A) BPE Tokenization Using a Pretrained Model

In [ ]:
!pip install -q transformers

from transformers import AutoTokenizer

# GPT-2 uses BPE-based tokenization
tokenizer_bpe_pretrained = AutoTokenizer.from_pretrained('gpt2')

tokens = tokenizer_bpe_pretrained.tokenize(text)
token_ids = tokenizer_bpe_pretrained.convert_tokens_to_ids(tokens)

print('Total number of tokens:', len(tokens))
print('\nToken\t\tToken ID')
print('-' * 40)

for token, token_id in zip(tokens, token_ids):
    print(f'{token}\t\t{token_id}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1348 > 1024). Running this sequence through the model will result in indexing errors


Total number of tokens: 1348

Token		Token ID
----------------------------------------
Natural		35364
Ġlanguage		3303
Ġprocessing		7587
Ġis		318
Ġa		257
Ġbranch		8478
Ġof		286
Ġartificial		11666
Ġintelligence		4430
Ġthat		326
Ċ		198
en		268
ables		2977
Ġcomputers		9061
Ġto		284
Ġunderstand		1833
,		11
Ġinterpret		6179
,		11
Ġand		290
Ġgenerate		7716
Ġhuman		1692
Ġlanguage		3303
.		13
Ċ		198
Natural		35364
Ġlanguage		3303
Ġprocessing		7587
Ġcombines		21001
Ġlingu		20280
istics		3969
,		11
Ġcomputer		3644
Ġscience		3783
,		11
Ġand		290
Ċ		198
machine		30243
Ġlearning		4673
Ġto		284
Ġprocess		1429
Ġlarge		1588
Ġamounts		6867
Ġof		286
Ġnatural		3288
Ġlanguage		3303
Ġdata		1366
.		13
Ċ		198
Ċ		198
Token		30642
ization		1634
Ġis		318
Ġone		530
Ġof		286
Ġthe		262
Ġfundamental		7531
Ġsteps		4831
Ġin		287
Ġnatural		3288
Ġlanguage		3303
Ġprocessing		7587
.		13
Ċ		198
A		32
Ġtoken		11241
izer		7509
Ġdivides		36319
Ġtext		2420
Ġinto		656
Ġsmaller		4833
Ġunits		4991
Ġcalled		1444
Ġtokens		16326
.		

### Test Pretrained BPE on a Sentence

In [ ]:
sentence = 'Natural language processing enables computers to understand human language.'

tokens = tokenizer_bpe_pretrained.tokenize(sentence)
token_ids = tokenizer_bpe_pretrained.convert_tokens_to_ids(tokens)

print('Sentence:')
print(sentence)
print('\nSubword Tokens and Token IDs:')
print('-' * 50)

for token, token_id in zip(tokens, token_ids):
    print(f'{token:20} {token_id}')

Sentence:
Natural language processing enables computers to understand human language.

Subword Tokens and Token IDs:
--------------------------------------------------
Natural              35364
Ġlanguage            3303
Ġprocessing          7587
Ġenables             13536
Ġcomputers           9061
Ġto                  284
Ġunderstand          1833
Ġhuman               1692
Ġlanguage            3303
.                    13


## 5.1(B) BPE Tokenization Without a Pretrained Model

In [ ]:
!pip install -q tokenizers

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer_bpe_custom = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer_bpe_custom.pre_tokenizer = Whitespace()

special_tokens = ['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]']

trainer = BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=special_tokens
)

tokenizer_bpe_custom.train_from_iterator([text], trainer=trainer)

print('BPE tokenizer trained successfully!')

output = tokenizer_bpe_custom.encode(text)
tokens = output.tokens
token_ids = output.ids

print('\nTotal number of tokens:', len(tokens))
print('\nToken\t\tToken ID')
print('-' * 40)

for token, token_id in zip(tokens, token_ids):
    print(f'{token}\t\t{token_id}')

BPE tokenizer trained successfully!

Total number of tokens: 1608

Token		Token ID
----------------------------------------
Natural		359
language		142
processing		192
is		82
a		24
b		25
r		41
an		54
ch		84
of		89
ar		59
t		43
ific		444
ial		380
in		51
t		43
el		105
l		35
ig		307
ence		178
that		266
en		50
ab		99
les		310
computer		463
s		42
to		62
understand		217
,		5
in		51
ter		90
pre		227
t		43
,		5
and		73
gener		350
ate		415
h		31
u		44
man		254
language		142
.		7
Natural		359
language		142
processing		192
com		107
b		25
in		51
es		57
linguistic		485
s		42
,		5
computer		463
s		42
c		26
i		32
ence		178
,		5
and		73
machine		338
learning		209
to		62
process		150
large		283
a		24
mo		122
unts		427
of		89
natural		226
language		142
data		292
.		7
Token		496
ization		175
is		82
one		414
of		89
the		68
f		29
un		76
d		27
a		24
mental		449
ste		273
ps		317
in		51
natural		226
language		142
processing		192
.		7
A		8
tokenizer		139
divid		215
es		57
text		200
into		185
smaller		291
units	

### Test Custom BPE Tokenizer

In [ ]:
test_sentence = '''internationalization computational communication
classification unbelievable preprocessing multilingual modernization understanding'''

output = tokenizer_bpe_custom.encode(test_sentence)

print('Input:')
print(test_sentence)
print('\nSubword Tokens:')
print(output.tokens)
print('\nToken IDs:')
print(output.ids)

Input:
internationalization computational communication
classification unbelievable preprocessing multilingual modernization understanding

Subword Tokens:
['internationalization', 'comput', 'ational', 'commun', 'ication', 'classification', 'un', 'be', 'li', 'ev', 'able', 'preprocessing', 'mult', 'ilingual', 'mo', 'dern', 'ization', 'understanding']

Token IDs:
[349, 462, 187, 440, 188, 355, 76, 79, 383, 302, 274, 352, 390, 473, 122, 456, 175, 472]


## 5.2(A) SentencePiece Tokenization Using a Pretrained Model

In [ ]:
!pip install -q sentencepiece transformers

from transformers import AutoTokenizer

# T5 uses a SentencePiece-based tokenizer
tokenizer_sp_pretrained = AutoTokenizer.from_pretrained('t5-small')

tokens = tokenizer_sp_pretrained.tokenize(text)
token_ids = tokenizer_sp_pretrained.convert_tokens_to_ids(tokens)

print('Total number of tokens:', len(tokens))
print('\nToken\t\tToken ID')
print('-' * 50)

for token, token_id in zip(tokens, token_ids):
    print(f'{token:25} {token_id}')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1319 > 512). Running this sequence through the model will result in indexing errors


Total number of tokens: 1319

Token		Token ID
--------------------------------------------------
▁Natural                  6869
▁language                 1612
▁processing               3026
▁is                       19
▁                         3
a                         9
▁branch                   6421
▁of                       13
▁artificial               7353
▁intelligence             6123
▁that                     24
▁                         3
enables                   7161
▁computers                7827
▁to                       12
▁understand               734
,                         6
▁interpret                7280
,                         6
▁and                      11
▁generate                 3806
▁human                    936
▁language                 1612
.                         5
▁Natural                  6869
▁language                 1612
▁processing               3026
▁                         3
combines                  15256
▁                         3
linguist

## 5.2(B) SentencePiece Tokenization Without a Pretrained Model

In [ ]:
import sentencepiece as spm

with open('/content/sp_dataset.txt', 'w', encoding='utf-8') as f:
    f.write(text)

spm.SentencePieceTrainer.train(
    input='/content/sp_dataset.txt',
    model_prefix='/content/my_sentencepiece',
    vocab_size=500,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor(model_file='/content/my_sentencepiece.model')

print('SentencePiece model trained successfully!')
print('Vocabulary size:', sp.get_piece_size())

SentencePiece model trained successfully!
Vocabulary size: 500


In [ ]:
tokens = sp.encode(text, out_type=str)
token_ids = sp.encode(text, out_type=int)

print('Total number of tokens:', len(tokens))
print('\nToken\t\tToken ID')
print('-' * 50)

for token, token_id in zip(tokens, token_ids):
    print(f'{token:25} {token_id}')

Total number of tokens: 1787

Token		Token ID
--------------------------------------------------
▁N                        214
atural                    175
▁language                 119
▁processing               179
▁is                       85
▁a                        8
▁b                        33
r                         459
an                        16
ch                        211
▁of                       59
▁ar                       400
t                         458
ific                      328
ial                       385
▁int                      157
el                        62
l                         463
ig                        299
ence                      154
▁that                     274
▁en                       322
ab                        74
les                       315
▁compu                    279
ter                       56
s                         461
▁to                       23
▁understand               208
,                         473
▁in          

## 5.3(A) POS Tagging Using SpaCy

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy

nlp = spacy.load('en_core_web_sm')
sentence = 'The young student is reading an interesting book in the library.'
doc = nlp(sentence)

print('{:<15} {:<10} {}'.format('Token', 'POS', 'Description'))
print('-' * 65)

seen_tokens = set()

for token in doc:
    if token.text.lower() not in seen_tokens:
        seen_tokens.add(token.text.lower())
        print(f'{token.text:<15} {token.pos_:<10} {spacy.explain(token.pos_)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 65.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Token           POS        Description
-----------------------------------------------------------------
The             DET        determiner
young           ADJ        adjective
student         NOUN       noun
is              AUX        auxiliary
reading         VERB       verb
an              DET        determiner
interesting     ADJ        adjective
book            NOUN       noun
in              ADP        adposition
library         NOUN       noun
.               PUNCT      punctuation


## 5.3(B) POS Tagging Using NLTK

In [ ]:
!pip install -q nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize
from nltk import pos_tag

sentence = 'The young student is reading an interesting book in the library.'
tokens = word_tokenize(sentence)
tagged_tokens = pos_tag(tokens)

pos_descriptions = {
    'CC': 'Coordinating conjunction',
    'CD': 'Cardinal number',
    'DT': 'Determiner',
    'IN': 'Preposition or subordinating conjunction',
    'JJ': 'Adjective',
    'JJR': 'Adjective, comparative',
    'JJS': 'Adjective, superlative',
    'MD': 'Modal',
    'NN': 'Noun, singular or mass',
    'NNS': 'Noun, plural',
    'NNP': 'Proper noun, singular',
    'PRP': 'Personal pronoun',
    'RB': 'Adverb',
    'TO': 'To',
    'VB': 'Verb, base form',
    'VBD': 'Verb, past tense',
    'VBG': 'Verb, gerund or present participle',
    'VBN': 'Verb, past participle',
    'VBP': 'Verb, non-3rd person singular present',
    'VBZ': 'Verb, 3rd person singular present',
    'WDT': 'Wh-determiner',
    'WP': 'Wh-pronoun',
    'WRB': 'Wh-adverb'
}

print('{:<15} {:<10} {}'.format('Token', 'POS', 'Description'))
print('-' * 80)

seen = set()
for token, tag in tagged_tokens:
    token_lower = token.lower()
    if token_lower not in seen:
        seen.add(token_lower)
        description = pos_descriptions.get(tag, 'Unknown POS tag')
        print(f'{token:<15} {tag:<10} {description}')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Token           POS        Description
--------------------------------------------------------------------------------
The             DT         Determiner
young           JJ         Adjective
student         NN         Noun, singular or mass
is              VBZ        Verb, 3rd person singular present
reading         VBG        Verb, gerund or present participle
an              DT         Determiner
interesting     JJ         Adjective
book            NN         Noun, singular or mass
in              IN         Preposition or subordinating conjunction
library         NN         Noun, singular or mass
.               .          Unknown POS tag


## 5.4 POS Tagging + Frequency Using SpaCy

In [ ]:
from collections import Counter

doc = nlp(text)

# Count frequencies of non-punctuation, non-space tokens
frequency = Counter()

for token in doc:
    if not token.is_punct and not token.is_space:
        frequency[token.text.lower()] += 1

# Store POS information for each unique token
token_information = {}

for token in doc:
    if not token.is_punct and not token.is_space:
        word = token.text.lower()
        if word not in token_information:
            token_information[word] = {
                'token': token.text,
                'pos': token.pos_,
                'description': spacy.explain(token.pos_),
                'frequency': frequency[word]
            }

print('{:<25} {:<10} {:<35} {:<10}'.format('Token', 'POS', 'Description', 'Frequency'))
print('-' * 85)

for word, information in sorted(
    token_information.items(),
    key=lambda x: (-x[1]['frequency'], x[0])
):
    print(
        f"{information['token']:<25}"
        f"{information['pos']:<10}"
        f"{information['description']:<35}"
        f"{information['frequency']:<10}"
    )

Token                     POS        Description                         Frequency 
-------------------------------------------------------------------------------------
the                      DET       determiner                         48        
a                        DET       determiner                         28        
and                      CCONJ     coordinating conjunction           27        
can                      AUX       auxiliary                          23        
of                       ADP       adposition                         22        
subword                  NOUN      noun                               17        
be                       AUX       auxiliary                          15        
to                       PART      particle                           15        
is                       AUX       auxiliary                          14        
as                       ADP       adposition                         13        
BPE                 

### Display POS Results as a DataFrame

In [ ]:
import pandas as pd

rows = []

for word, information in token_information.items():
    rows.append({
        'Token': information['token'],
        'POS': information['pos'],
        'Description': information['description'],
        'Frequency': information['frequency']
    })

df = pd.DataFrame(rows)
df = df.sort_values(by='Frequency', ascending=False).reset_index(drop=True)
df

,Token,POS,Description,Frequency
0,the,DET,determiner,48
1,a,DET,determiner,28
2,and,CCONJ,coordinating conjunction,27
3,can,AUX,auxiliary,23
4,of,ADP,adposition,22
...,...,...,...,...
360,modernization,NOUN,noun,1
361,purpose,NOUN,noun,1
362,learns,VERB,verb,1
363,reusable,ADJ,adjective,1


## Conclusion

In this experiment, subword-level tokenization was implemented using BPE and SentencePiece, both with pretrained tokenizers and with tokenizers trained on the supplied corpus. The generated subword tokens and their token IDs were examined. POS tagging was also performed using SpaCy and NLTK, and SpaCy was used to identify unique tokens, their POS descriptions, and frequencies.